# Silver Layer Autoloader
This notebook reads new bronze JSON files as they arrive and writes them into a silver Delta table using Databricks Autoloader.

In [ ]:
from pathlib import PurePosixPath
import sys

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
repo_root = PurePosixPath("/Workspace") / PurePosixPath(notebook_path).parents[2]
sys.path.append(str(repo_root))

In [ ]:
bronze_path = "/Volumes/openaq/bronze_dev/sensor_data"
silver_path = "/Volumes/openaq/silver_dev/sensor_data"
checkpoint_path = "/Volumes/openaq/silver_dev/_checkpoints/sensor_data_autoloader"
schema_location = "/Volumes/openaq/silver_dev/_schema/sensor_data_autoloader"

print(f"bronze_path = {bronze_path}")
print(f"silver_path = {silver_path}")
print(f"checkpoint_path = {checkpoint_path}")
print(f"schema_location = {schema_location}")

In [ ]:
bronze_sample_df = spark.read.json(f"{bronze_path}/*")
bronze_sample_df.printSchema()
display(bronze_sample_df.limit(5))

In [ ]:
%sql
USE CATALOG openaq;
CREATE SCHEMA IF NOT EXISTS silver_dev;
CREATE TABLE IF NOT EXISTS silver_dev.sensor_data
USING DELTA
LOCATION '/Volumes/openaq/silver_dev/sensor_data';

In [ ]:
streaming_df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", schema_location)
        .option("cloudFiles.inferColumnTypes", "true")
        .option("recursiveFileLookup", "true")
        .load(bronze_path)
)

query = (
    streaming_df.writeStream
        .format("delta")
        .option("checkpointLocation", checkpoint_path)
        .option("mergeSchema", "true")
        .outputMode("append")
        .table("openaq.silver_dev.sensor_data")
)

query

In [ ]:
%sql
SELECT count(*) AS total_rows
FROM openaq.silver_dev.sensor_data;

In [ ]:
%sql
SELECT *
FROM openaq.silver_dev.sensor_data
LIMIT 20;